PMR3508-2026-24
Classification of Adult Database (from Kaggle) using k-Nearest Neighbor (kNN)

This Notebook is divided in the following sections:

Classifier Development
Exploratory Analysis

First, we import the libraries and the data that we're going to use

In [516]:
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

In [517]:
adult_data = pd.read_csv("adult.data",
        names=[
        "Age", "Workclass", "fnlwgt", "Education", "Education-Num", "Martial Status",
        "Occupation", "Relationship", "Race", "Sex", "Capital Gain", "Capital Loss",
        "Hours per week", "Country", "Target"],
        sep=r'\s*,\s*',
        engine='python',
        na_values="?")

In [518]:
adult_data.shape


(32561, 15)

In [519]:
adult_data.head()

,Age,Workclass,fnlwgt,Education,Education-Num,Martial Status,Occupation,Relationship,Race,Sex,Capital Gain,Capital Loss,Hours per week,Country,Target
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [520]:
adult_test = pd.read_csv("adult.test",skiprows=1,
        names=[
        "Age", "Workclass", "fnlwgt", "Education", "Education-Num", "Martial Status",
        "Occupation", "Relationship", "Race", "Sex", "Capital Gain", "Capital Loss",
        "Hours per week", "Country", "Target"],
        sep=r'\s*,\s*',
        engine='python',
        na_values="?")

In [521]:
adult_test.shape

(16281, 15)

In [522]:
adult_test.head()

,Age,Workclass,fnlwgt,Education,Education-Num,Martial Status,Occupation,Relationship,Race,Sex,Capital Gain,Capital Loss,Hours per week,Country,Target
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K.
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K.
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K.
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K.
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,<=50K.


In [523]:
adult_test["Target"].value_counts()

Target
<=50K.    12435
>50K.      3846
Name: count, dtype: int64

In [524]:
adult_test["Target"] = adult_test["Target"].str.rstrip(".")

In [525]:
adult_test["Target"].value_counts()

Target
<=50K    12435
>50K      3846
Name: count, dtype: int64

In [526]:
adult_data.isna().sum()

Age                  0
Workclass         1836
fnlwgt               0
Education            0
Education-Num        0
Martial Status       0
Occupation        1843
Relationship         0
Race                 0
Sex                  0
Capital Gain         0
Capital Loss         0
Hours per week       0
Country            583
Target               0
dtype: int64

In [527]:
adult_clean = adult_data.dropna()
test_clean = adult_test.dropna()

In [528]:
adult_clean.shape

(30162, 15)

In [529]:
test_clean.shape

(15060, 15)

In [530]:
adult_data_treinamento = adult_clean.drop(columns = "Target")
adult_data_resposta = adult_clean["Target"]
adult_test_avaliação = test_clean.drop(columns = "Target")
adult_test_resposta= test_clean["Target"]


In [531]:
adult_data_treinamento.shape

(30162, 14)

In [532]:
adult_data_resposta.shape

(30162,)

In [533]:
adult_test_avaliação.shape

(15060, 14)

In [534]:
adult_test_resposta.shape

(15060,)

In [535]:
adult_data_treinamento.dtypes

Age                int64
Workclass         object
fnlwgt             int64
Education         object
Education-Num      int64
Martial Status    object
Occupation        object
Relationship      object
Race              object
Sex               object
Capital Gain       int64
Capital Loss       int64
Hours per week     int64
Country           object
dtype: object

In [536]:
colunas_numericas = ["Age", "fnlwgt", "Education-Num", "Capital Gain", "Capital Loss", "Hours per week"]
colunas_categoricas = ["Workclass", "Education", "Martial Status", "Occupation", "Relationship", "Race", "Sex", "Country"]

In [537]:
len(colunas_numericas)

6

In [538]:
len(colunas_categoricas)

8

In [539]:
preprocessador = ColumnTransformer([("final_numbers", StandardScaler(), colunas_numericas), ("final_categorics", OneHotEncoder(handle_unknown="ignore"), colunas_categoricas)])

In [540]:
Xadult = preprocessador.fit_transform(adult_data_treinamento)
Xadult.shape

(30162, 104)

In [541]:
model = [("prep", preprocessador), ("knn", KNeighborsClassifier(n_neighbors= 5))]
sleep = Pipeline(model)

In [542]:
resultado_final = cross_val_score(sleep, adult_data_treinamento, adult_data_resposta, cv=5).mean()
print(resultado_final)

0.8283273835197308


In [543]:
candidatos = [1, 5, 11, 15, 21, 25, 31, 35, 41, 45, 51, 55, 61]
parametros = {"knn__n_neighbors": candidatos}
busca = GridSearchCV(sleep, parametros, cv=5, n_jobs = -1)

In [544]:
busca.fit(adult_data_treinamento, adult_data_resposta)

KeyboardInterrupt: 

In [ ]:
print(busca.best_params_)
print(busca.best_score_)